In [28]:
import pandas as pd
df =pd.read_excel("./1.xlsx",encoding='gbk')

In [29]:
#keywords=['25780187430676','25290298109676','25404794257676','25509416019676','25780197425676']
 #10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
#df['商品ID'].isin(keywords)
#df['商品ID'] = df['商品ID'].astype(str)


#删除其他行
#df.drop(index= df[ df['商品ID'].isin(keywords)==False].index,inplace=True)
#df.reset_index(drop=True,inplace= True)

In [30]:
df["实付款"] = df["实付款"].astype(str).str.replace("¥", "").astype(float)

In [31]:
from datetime import datetime
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单创建时间"] = df["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码"},inplace=True)
df1 = df1[["SKU编码","产品名称"]]
df= pd.merge(df,df1,on="SKU编码",how="left")

In [32]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码'].value_counts()
lackCode

Series([], Name: SKU编码, dtype: int64)

In [33]:
lackCode.to_excel('缺码表.xlsx')

In [34]:
df['订单数'] = df['成交数量']

In [35]:
df["下单单量"] = df['订单数']

In [36]:
df['发货时长（小时）'] = pd.to_datetime(df['发货时间']) - pd.to_datetime(df['订单支付时间'])

In [37]:
df['发货时长（小时）'] = df['发货时长（小时）'].dt.total_seconds() / 3600

In [38]:
df['发货时长（小时）'] = df['发货时长（小时）'].clip(lower=0)

In [39]:
df['发货时长（小时）']

0            NaN
1       0.249167
2            NaN
3            NaN
4       0.330833
5            NaN
6            NaN
7       0.421111
8            NaN
9       0.507500
10      0.573611
11      0.614444
12      0.724444
13           NaN
14      0.955000
15      0.033056
16      0.050833
17      0.057778
18      0.067778
19      0.084167
20      0.085833
21           NaN
22      0.245556
23      0.257778
24      0.262500
25      0.292778
26      0.517500
27           NaN
28           NaN
29      0.783333
          ...   
2175    0.572500
2176    0.570833
2177    0.587222
2178    0.672222
2179    0.700278
2180    0.728611
2181    0.742222
2182    0.754167
2183    0.826667
2184    0.828889
2185    0.847222
2186         NaN
2187    0.926944
2188    0.930833
2189    0.934722
2190    0.963611
2191    0.965556
2192    0.975556
2193    0.996667
2194    1.078056
2195    1.105278
2196    1.116389
2197         NaN
2198    1.131389
2199         NaN
2200         NaN
2201    1.197778
2202         N

In [40]:
# 清除未支付订单
from datetime import datetime
df["订单支付时间"] = df["订单支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["订单支付时间"] = pd.to_datetime(df["订单支付时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单支付时间"] = df["订单支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [41]:
df["发货时间"] = df["发货时间"].fillna(pd.to_datetime('1970-01-01'))
df["发货时间"] = pd.to_datetime(df["发货时间"],format = "%Y-%m-%d %H:%M:%S")
df["发货时间"] = df["发货时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [42]:
df["支付单量"] = 0
df["发货单量"] = 0
df["支付金额"] = 0
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01":
        df.loc[i,"发货单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'实付款']

In [43]:
df["售后状态"] = df["售后状态"].astype(str)

In [44]:
df["商品ID"] = df["商品ID"].astype(str)

In [45]:
df["售后状态"].value_counts()

nan      1713
退款成功      427
待卖家收货      42
待买家退货      16
退款关闭        6
待卖家处理       1
Name: 售后状态, dtype: int64

In [46]:
df["商品ID"].value_counts()

26916829094676    1312
26878682641676     748
26878898825676      60
26929532780676      21
26929339506676      14
26919842811676      14
26877137670676      12
26916921062676      10
26878816202676       5
26917016128676       2
26878634264676       2
26916921102676       2
26919948274676       2
26919947999676       1
Name: 商品ID, dtype: int64

In [47]:
df['物流信息'].value_counts()

已签收      964
未知       313
商家已接单    249
运输中      247
已揽件      185
待取件      147
已发货       64
派送中       21
已退签        9
物流异常       3
退回中        2
收件人拒签      1
Name: 物流信息, dtype: int64

In [48]:
for i in range (0,len(df)):
    if df.loc[i,"订单状态"] == "已收货" and (df.loc[i,'售后状态'] == 'nan' or df.loc[i,'售后状态'] == '退款关闭'):
        df.loc[i,"订单状态"] = "交易成功"

In [49]:
df["已签收"] = 0
df["发货中"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收数量"] = 0
df["发货前退款单量"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "交易成功" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"已签收"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已发货" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"发货中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"售后状态"] == "待买家处理" or df.loc[i,"售后状态"] == "待买家退货" or df.loc[i,"售后状态"] == "待卖家处理" or df.loc[i,"售后状态"]=="待卖家收货" or df.loc[i,"售后状态"]=="待平台判定":
        df.loc[i,"售后中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01"  and  df.loc[i,"发货时间"] == "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"预计签收单量"] = df.loc[i,"订单数"]

In [50]:
import numpy as np
result = pd.pivot_table(df,index=['订单创建时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额','发货时长（小时）'] , aggfunc = np.sum)
result = result.reset_index()
result = result[['订单创建时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额','发货时长（小时）']]

In [51]:
# 拼接店铺消耗表
df_consume = pd.read_excel("快手店铺消耗13号.xlsx")
df_consume = df_consume[['日期','产品','花费','转化数']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['订单创建时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
df_consume.rename(columns={"产品":"产品名称"},inplace=True)
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['订单创建时间','产品名称'] , values = ['花费','转化数'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["订单创建时间"] = pd.to_datetime(df_consume["订单创建时间"],format = "%Y-%m-%d")
df_consume["订单创建时间"] = df_consume["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [52]:
final = pd.merge(result,df_consume,on = ['订单创建时间','产品名称'],how = 'left')
final['发货时长'] = final['发货时长（小时）'] / final['支付单量']
final = final[['订单创建时间',"产品名称",'转化数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额','发货时长','发货时长（小时）']]
final.rename(columns={"转化数":"总广告单量"},inplace=True)

In [53]:
final

,订单创建时间,产品名称,总广告单量,下单单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,支付金额,发货时长,发货时长（小时）
0,2026-10-05,8688#塑形美体发热保暖裤,203.0,264,208,176,144,0,1,31,32,144.0,8409.2,1.760984,366.284722
1,2026-10-06,8688#塑形美体发热保暖裤,94.0,120,98,77,65,0,0,12,21,65.0,3930.2,1.695720,166.180556
2,2026-10-07,8688#塑形美体发热保暖裤,67.0,79,66,53,40,1,0,12,13,41.0,2653.4,1.300787,85.851944
3,2026-10-08,8688#塑形美体发热保暖裤,35.0,43,33,28,22,1,0,5,5,23.0,1336.7,1.937365,63.933056
4,2026-10-09,8688#塑形美体发热保暖裤,74.0,86,74,58,47,0,1,10,16,47.0,3012.6,1.464441,108.368611
5,2026-10-10,8688#塑形美体发热保暖裤,29.0,39,29,26,17,0,0,9,3,17.0,1177.1,1.194598,34.643333
6,2026-10-11,8688#塑形美体发热保暖裤,32.0,43,32,27,18,0,1,8,5,18.0,1316.8,1.031866,33.019722
7,2026-10-12,8688#塑形美体发热保暖裤,33.0,42,33,27,23,1,1,2,6,24.0,1336.7,2.260522,74.597222
8,2026-10-13,8688#塑形美体发热保暖裤,29.0,37,29,23,18,1,0,4,6,19.0,1177.1,1.983602,57.524444
9,2026-10-14,8688#塑形美体发热保暖裤,37.0,38,37,34,22,0,2,10,3,22.0,1476.3,1.076179,39.818611


In [54]:
final.to_excel('直播签收汇总.xlsx')